In [1]:
import torch
import gc
import wandb
import warnings
import optuna

import torch.nn.functional as F
import torch.nn as nn
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score
from torch.optim.lr_scheduler import StepLR

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
class TextRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        print("Pre-tokenizing dataset (this takes a minute)...")
        
        cv_texts = dataframe['cv_text'].astype(str).tolist()
        vac_texts = dataframe['vacancy_text'].astype(str).tolist()
        
        # Tokenize everything upfront
        self.cv_encodings = tokenizer(
            cv_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )
        self.vac_encodings = tokenizer(
            vac_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )

        self.labels = dataframe['response'].values
        self.cvids = dataframe['cvid'].astype(str).tolist()
        self.vacancy_ids = dataframe['humanjobid'].astype(str).tolist()
        
        # --- THE CORE DIFFERENCE: GROUP BY CANDIDATE ---
        self.unique_cvids = dataframe['cvid'].unique().tolist()
        self.grouped_indices = dataframe.groupby('cvid').indices

    def __len__(self):
        # Length is now the number of unique candidates, NOT the number of total rows
        return len(self.unique_cvids)

    def __getitem__(self, idx):
        # 1. Look up the candidate
        cvid = self.unique_cvids[idx]
        # 2. Get the specific row indices for all N of their vacancies
        indices = self.grouped_indices[cvid] 
        
        # 3. Return the N-sized chunk of tensors for this single candidate
        return {
            'cv_input_ids': self.cv_encodings['input_ids'][indices],
            'cv_attention_mask': self.cv_encodings['attention_mask'][indices],
            'vac_input_ids': self.vac_encodings['input_ids'][indices],
            'vac_attention_mask': self.vac_encodings['attention_mask'][indices],
            'labels': torch.tensor(self.labels[indices], dtype=torch.float32),
            'cvid': [self.cvids[i] for i in indices],
            'vacancy_id': [self.vacancy_ids[i] for i in indices]
        }

def ranking_collate_fn(batch):
    """
    Takes a list of candidate dictionaries (where each dict contains N items)
    and concatenates them along the 0th dimension to mimic PyG batching.
    """
    return {
        'cv_input_ids': torch.cat([b['cv_input_ids'] for b in batch], dim=0),
        'cv_attention_mask': torch.cat([b['cv_attention_mask'] for b in batch], dim=0),
        'vac_input_ids': torch.cat([b['vac_input_ids'] for b in batch], dim=0),
        'vac_attention_mask': torch.cat([b['vac_attention_mask'] for b in batch], dim=0),
        'labels': torch.cat([b['labels'] for b in batch], dim=0),
        
        # Flatten the nested lists of strings
        'cvid': [c for b in batch for c in b['cvid']],
        'vacancy_id': [v for b in batch for v in b['vacancy_id']]
    }

In [3]:
trainloader = torch.load(f'../dataloaders/sentence_trainloader.pth',
                         weights_only=False)
valloader = torch.load(f'../dataloaders/sentence_valloader.pth',
                         weights_only=False)
testloader = torch.load(f'../dataloaders/sentence_testloader.pth',
                         weights_only=False)

In [4]:
import torch.nn.functional as F

def multiple_negatives_ranking_loss(cv_embeddings, vac_embeddings, temperature=0.05):
    """
    Computes contrastive loss. The diagonal contains the positive 'next sentences'.
    Everything else is an in-batch negative.
    """
    # Normalize embeddings so dot product equals cosine similarity
    cv_embeddings = F.normalize(cv_embeddings, p=2, dim=1)
    vac_embeddings = F.normalize(vac_embeddings, p=2, dim=1)
    
    # Compute similarity matrix: [batch_size, batch_size]
    scores = torch.matmul(cv_embeddings, vac_embeddings.T) / temperature
    
    # The true "next sentence" for cv[i] is vac[i], which is on the diagonal
    labels = torch.arange(scores.size(0), device=scores.device)
    
    # Standard Cross Entropy forces the diagonal toward 1 and off-diagonals toward 0
    return F.cross_entropy(scores, labels)

In [5]:
class text_ranker(torch.nn.Module):
    def __init__(self, pooling="mean"):
        super().__init__()
        self.model = AutoModel.from_pretrained("jjzha/dajobbert-base-uncased")
        self.pooling = pooling
        
        # Freeze the bottom 8 layers for speed
        for name, param in self.model.named_parameters():
            if 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < 8:  
                    param.requires_grad = False

    def pool_embeddings(self, outputs, attention_mask):
        if self.pooling == "mean":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            sum_embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) 
            return sum_embeddings / sum_mask
            
        elif self.pooling == "sum":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            return torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
        
        elif self.pooling == "max":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).bool()
            masked_embeddings = outputs.last_hidden_state * input_mask_expanded 
            embeddings, _ = torch.max(masked_embeddings, dim=1) 
            return embeddings

    def forward(self, batch_cv, batch_vac):
        cv_outputs = self.model(batch_cv['input_ids'], attention_mask=batch_cv['attention_mask'])
        cv_emb = self.pool_embeddings(cv_outputs, batch_cv['attention_mask'])
        
        vac_outputs = self.model(batch_vac['input_ids'], attention_mask=batch_vac['attention_mask'])
        vac_emb = self.pool_embeddings(vac_outputs, batch_vac['attention_mask'])
        
        # Return the raw representations
        return cv_emb, vac_emb

In [6]:
def train_loop(model, optimizer, trainloader, use_wandb=False):
    model.train()
    scaler = torch.cuda.amp.GradScaler() 
    epoch_losses = []
        
    for i, batch in enumerate(trainloader):
        batch_cv = {
            'input_ids': batch['cv_input_ids'].to(device),
            'attention_mask': batch['cv_attention_mask'].to(device)
        }
        batch_vac = {
            'input_ids': batch['vac_input_ids'].to(device),
            'attention_mask': batch['vac_attention_mask'].to(device)
        }

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            # Get the representations
            cv_emb, vac_emb = model(batch_cv, batch_vac)
            
            # Compute MNRL (InfoNCE)
            loss = multiple_negatives_ranking_loss(cv_emb, vac_emb)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        scaler.step(optimizer)
        scaler.update()

        epoch_losses.append(loss.item())
        print(f"Batch: {i + 1}/{len(trainloader)}, Loss: {loss.item():.4f}    ", end="\r", flush=True)
        
    return epoch_losses


def val_loop(model, valloader):
    model.eval() # Switch to evaluation mode
    ndcg_scores = []
    
    with torch.no_grad():
        for i, batch in enumerate(valloader):
            print(f"Batch: {i + 1}/{len(valloader)}              ", end="\r")
            
            batch_cv = {
                'input_ids': batch['cv_input_ids'].to(device),
                'attention_mask': batch['cv_attention_mask'].to(device)
            }
            batch_vac = {
                'input_ids': batch['vac_input_ids'].to(device),
                'attention_mask': batch['vac_attention_mask'].to(device)
            }
            ground_truth = batch['labels'].to(device)
            
            # 1. Get the raw embeddings from the Bi-Encoder
            with torch.cuda.amp.autocast():
                cv_emb, vac_emb = model(batch_cv, batch_vac)
                
                # 2. Calculate the cosine similarity for the pairs
                # This becomes your "prediction score" for NDCG
                y_pred_val = F.cosine_similarity(cv_emb, vac_emb)
                
            if len(y_pred_val) > len(ground_truth):
                y_pred_val = y_pred_val[:len(ground_truth)]
        
            # 3. Evaluate using the exact same NDCG logic as before
            score = ndcg_score(ground_truth.detach().cpu().unsqueeze(0), 
                               y_pred_val.squeeze().unsqueeze(0).detach().cpu(), k=10)
            ndcg_scores.append(score)
            
    return ndcg_scores

In [7]:
def train_model(trial, trainloader, valloader, epochs=5):
    best_score = 0
    
    # Search space
    learning_rate = trial.suggest_float('learning_rate', 1e-6, 1e-4, log=True)
    pooling_method = trial.suggest_categorical('pooling_method', ["mean", "max", "sum"])

    model = text_ranker(pooling=pooling_method).to(device)      
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = StepLR(optimizer, step_size=3, gamma=0.1)
    
    for epoch in range(epochs + 1):
        print(f"Epoch: {epoch}/{epochs}")

        # Train the model for the current epoch (Returns MNRL Loss)
        epoch_losses = train_loop(model, optimizer, trainloader, use_wandb)

        # --- FIX: Correctly label the output as Loss ---
        print(f"\nTraining Loss (MNRL): {np.mean(epoch_losses):.4f}\n")
        scheduler.step()

        if use_wandb:
            wandb.log({"Training Loss": np.mean(epoch_losses)})

        # Evaluate the model (Returns NDCG)
        val_ndcg_scores = val_loop(model, valloader)
        print(f"\nTesting nDCG: {np.mean(val_ndcg_scores):.4f}\n")

        if np.mean(val_ndcg_scores) > best_score:
            best_score = np.mean(val_ndcg_scores)
            
    return best_score

In [8]:
def objective_wrapper(trainloader, valloader):
    def objective(trial):
        # Make sure this calls train_model, NOT train_loop!
        return train_model(trial, trainloader, valloader) 
    
    return objective

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_wandb = False # Define this so it doesn't crash during the train loop check

torch.cuda.empty_cache() 
gc.collect()

# Hide user/future warnings
warnings.filterwarnings('ignore')

# Define the Optuna study
study = optuna.create_study(direction='maximize')

# We need to provide trainloader and valloader to the training/validation loop
wrapped_objective = objective_wrapper(trainloader, valloader)

# Start optimization
study.optimize(wrapped_objective, n_trials=5)  

print("Best hyperparameters:", study.best_trial.params)

[I 2026-07-22 16:37:09,381] A new study created in memory with name: no-name-77845b09-dc99-4a63-a916-7f0344e93752


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0/5
Batch: 276/276, Loss: 0.6973    
Training Loss (MNRL): 0.9235

Batch: 41/41              
Testing nDCG: 0.3745

Epoch: 1/5
Batch: 276/276, Loss: 1.3896    
Training Loss (MNRL): 0.9232

Batch: 41/41              
Testing nDCG: 0.3945

Epoch: 2/5
Batch: 276/276, Loss: 1.3838    
Training Loss (MNRL): 0.9217

Batch: 41/41              
Testing nDCG: 0.3802

Epoch: 3/5
Batch: 276/276, Loss: 2.0732    
Training Loss (MNRL): 0.9213

Batch: 41/41              
Testing nDCG: 0.3694

Epoch: 4/5
Batch: 276/276, Loss: 0.0000    
Training Loss (MNRL): 0.9216

Batch: 41/41              
Testing nDCG: 0.3584

Epoch: 5/5
Batch: 276/276, Loss: 1.0970    
Training Loss (MNRL): 0.9214

Batch: 39/41              

[I 2026-07-22 16:41:56,614] Trial 0 finished with value: 0.39452754676173435 and parameters: {'learning_rate': 5.3361038920905594e-05, 'pooling_method': 'mean'}. Best is trial 0 with value: 0.39452754676173435.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0/5
Batch: 276/276, Loss: 0.0000    
Training Loss (MNRL): 0.9310

Batch: 41/41              
Testing nDCG: 0.3964

Epoch: 1/5
Batch: 276/276, Loss: 1.3948    
Training Loss (MNRL): 0.9283

Batch: 41/41              
Testing nDCG: 0.3806

Epoch: 2/5
Batch: 276/276, Loss: 1.3926    
Training Loss (MNRL): 0.9230

Batch: 41/41              
Testing nDCG: 0.3874

Epoch: 3/5
Batch: 276/276, Loss: 1.0941    
Training Loss (MNRL): 0.9226

Batch: 41/41              
Testing nDCG: 0.3857

Epoch: 4/5
Batch: 276/276, Loss: 0.0000    
Training Loss (MNRL): 0.9226

Batch: 41/41              
Testing nDCG: 0.3861

Epoch: 5/5
Batch: 276/276, Loss: 1.6160    
Training Loss (MNRL): 0.9227

Batch: 41/41              

[I 2026-07-22 17:39:57,587] Trial 4 finished with value: 0.3964464086296742 and parameters: {'learning_rate': 4.627580794096919e-06, 'pooling_method': 'mean'}. Best is trial 4 with value: 0.3964464086296742.



Testing nDCG: 0.3861

Best hyperparameters: {'learning_rate': 4.627580794096919e-06, 'pooling_method': 'mean'}
